# Ovarian co-culture data: download, assemble and verify

Rebuilds `dataset_co_mo2.h5ad` from public data and checks it.

1. **Download** the raw single-cell counts of GEO series
   [GSE224333](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE224333)
   (ovarian clear-cell carcinoma cells and fibroblasts, mono- and co-culture).
2. **Re-attach** the authors' cell annotations stored in this folder
   (`dataset_co_mo2_cell_metadata.csv`: BatchID, CellType, ClusterID, CellCycle;
   `dataset_co_mo2_genes.txt`: gene order), joining on the cell barcode within each sample.
3. **Verify** cell numbers, total counts and the joint histograms of the QuantumXCT gene panel
   against the stored reference `data_prep/reference_histograms.csv` (must match exactly).

**Output:** `python/dataset_co_mo2.h5ad` (raw counts in `X`). Figures: `results/data/`.
Needs internet and `GEOparse` (`pip install GEOparse`). For the R tools, convert with
`data_prep/convert_h5ad_rds.R`.

In [ ]:
# Work from the QuantumXCT/python folder wherever Jupyter was started: the dataset is
# written to python/ and the figures to python/results/data/.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("working directory:", _root)

In [ ]:
import gzip, re, shutil, tempfile, warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # scanpy 1.13 moved read_10x_mtx to scanpy.io
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

GSE = "GSE224333"
OUT = "dataset_co_mo2.h5ad"                                          # in python/
META = os.path.join("data_prep", "dataset_co_mo2_cell_metadata.csv")
GENES = os.path.join("data_prep", "dataset_co_mo2_genes.txt")
REFERENCE = os.path.join("data_prep", "reference_histograms.csv")
WORKDIR = os.path.join(tempfile.gettempdir(), "gse224333")           # GEO download cache
REBUILD = True        # False: skip the download and only verify an existing dataset_co_mo2.h5ad
MAP_OVERRIDES = {}    # e.g. {"GSM7019822": "Cancer Cells (Mo)"} if step 1 mis-assigns a sample
FIG_DIR = os.path.join("results", "data")
os.makedirs(FIG_DIR, exist_ok=True)

# the four groups, as BatchID values in the stored annotations
BATCHES = ["Cancer Cells (Mo)", "Cancer Cells (Co)", "Fibroblasts (Mo)", "Fibroblasts (Co)"]

In [ ]:
def classify_title(title):
    """Infer a BatchID from a GEO sample title by keyword; None if unsure."""
    t = title.lower()
    if any(k in t for k in ("fibroblast", "caf", "stroma")):
        lineage = "Fibroblasts"
    elif any(k in t for k in ("cancer", "tumor", "tumour", "carcinoma", "occc", "epithel")):
        lineage = "Cancer Cells"
    else:
        return None
    if re.search(r"\b(co[-\s]?cult|coculture|co\b|together|mixed)", t):
        state = "Co"
    elif re.search(r"\b(mono|monocult|alone|single|solo)\b", t):
        state = "Mo"
    else:
        return None
    return f"{lineage} ({state})"


def read_10x_sample(files, tmp):
    """Read one sample's 10x matrix / barcodes / features files (any naming, gzipped or not)."""
    picks = {"matrix": None, "barcodes": None, "features": None}
    for f in files:
        low = os.path.basename(f).lower()
        if low.endswith(".mtx") or low.endswith(".mtx.gz"):
            picks["matrix"] = f
        elif "barcode" in low:
            picks["barcodes"] = f
        elif "feature" in low or "gene" in low:
            picks["features"] = f
    if not all(picks.values()):
        raise RuntimeError(f"no 10x triplet among: {[os.path.basename(x) for x in files]}")
    d = tempfile.mkdtemp(dir=tmp)
    for std, src in (("matrix.mtx.gz", picks["matrix"]), ("barcodes.tsv.gz", picks["barcodes"]),
                     ("features.tsv.gz", picks["features"])):
        dst = os.path.join(d, std)
        if src.endswith(".gz"):
            shutil.copy(src, dst)
        else:
            with open(src, "rb") as fi, gzip.open(dst, "wb") as fo:
                shutil.copyfileobj(fi, fo)
    a = sc.read_10x_mtx(d, var_names="gene_symbols", make_unique=True)
    a.var_names_make_unique()
    return a

## 1. Samples in the series

Lists every GEO sample and the group it is assigned to. Check this table: each of the four
groups must get exactly one sample. If a title is misread, set `MAP_OVERRIDES` above and re-run.

In [ ]:
if REBUILD:
    import GEOparse
    os.makedirs(WORKDIR, exist_ok=True)
    gse = GEOparse.get_GEO(geo=GSE, destdir=WORKDIR, silent=True)
    rows = []
    for gsm_id, gsm in gse.gsms.items():
        title = gsm.metadata.get("title", [""])[0]
        files = gsm.metadata.get("supplementary_file", []) + gsm.metadata.get("supplementary_file_1", [])
        rows.append({"GSM": gsm_id, "title": title,
                     "group": MAP_OVERRIDES.get(gsm_id) or classify_title(title),
                     "files": ", ".join(os.path.basename(u) for u in files)})
    samples = pd.DataFrame(rows)
    assigned = {b: samples.loc[samples.group == b, "GSM"].tolist() for b in BATCHES}
    problems = [b for b, g in assigned.items() if len(g) != 1] + samples.loc[samples.group.isna(), "GSM"].tolist()
    display(samples)
    if problems:
        raise ValueError(f"fix the sample assignment with MAP_OVERRIDES: {problems}")
    print("each group maps to one sample:", {b: g[0] for b, g in assigned.items()})
else:
    print(f"REBUILD = False -> using the existing {OUT}")

## 2. Download and assemble

Each group is joined only with its own sample: the barcodes have no sample suffix, so the
same barcode can occur in two samples. Cells and genes end up in the stored annotation order;
a stored gene missing from a sample is filled with zeros.

In [ ]:
if REBUILD:
    gse.download_supplementary_files(directory=WORKDIR, download_sra=False)

    meta = pd.read_csv(META)
    bc_col = "CellID" if "CellID" in meta.columns else meta.columns[0]
    meta[bc_col] = meta[bc_col].astype(str)
    meta = meta.set_index(bc_col, drop=False)
    genes = [g.strip() for g in open(GENES) if g.strip()]
    print(f"stored annotations: {len(meta)} cells, {len(genes)} genes")

    blocks, obs_order = [], []
    for b in BATCHES:
        gsm_id = assigned[b][0]
        files = [os.path.join(r, f) for r, _d, fs in os.walk(WORKDIR) for f in fs if gsm_id in f]
        a = read_10x_sample(files, WORKDIR)
        a.obs_names = [str(x) for x in a.obs_names]
        want = meta.index[meta["BatchID"] == b].tolist()
        missing = sorted(set(want) - set(a.obs_names))
        print(f"  {b:18} {gsm_id}: {a.n_obs} cells in GEO, {len(want) - len(missing)}/{len(want)} annotated matched")
        if missing:
            raise ValueError(f"{len(missing)} annotated barcodes not in {gsm_id}, e.g. {missing[:5]}")
        sub = a[pd.Index(want)]
        M = sub.X.tocsc()
        pos = {g: i for i, g in enumerate(sub.var_names)}
        src = [pos.get(g) for g in genes]
        dst = [j for j, c in enumerate(src) if c is not None]
        Xg = sp.lil_matrix((sub.n_obs, len(genes)), dtype=np.float32)
        if dst:
            Xg[:, dst] = M[:, [c for c in src if c is not None]]
        if len(dst) < len(genes):
            print(f"      {len(genes) - len(dst)} stored genes absent in this sample -> zeros")
        blocks.append(Xg.tocsr())
        obs_order.extend(want)

    obs = meta.loc[obs_order].drop(columns=["barcode"], errors="ignore")
    obs.index.name = None
    adata = ad.AnnData(X=sp.vstack(blocks).tocsr(), obs=obs, var=pd.DataFrame(index=genes))
    adata = adata[meta.index.tolist()].copy()          # cells in the stored annotation order
    for c in ("BatchID", "CellType", "CellCycle"):
        if c in adata.obs:
            adata.obs[c] = adata.obs[c].astype("category")
    assert adata.n_obs == len(meta) and adata.n_vars == len(genes)
    adata.write_h5ad(OUT)
    print(f"wrote {OUT}: {adata.n_obs} cells x {adata.n_vars} genes")

## 3. Verify

Gene panel: cancer cells = `PDGFB, STAT3, HES4, TGFB1`; fibroblasts = `PDGFRB, TGFBR1, TGFBR2`.

In [ ]:
d = ad.read_h5ad(OUT)
found = set(d.obs["BatchID"].astype(str))
assert set(BATCHES) <= found, f"missing groups: {set(BATCHES) - found}"
PANEL = {"ct1": ["PDGFB", "STAT3", "HES4", "TGFB1"], "ct2": ["PDGFRB", "TGFBR1", "TGFBR2"]}
missing = [g for g in PANEL["ct1"] + PANEL["ct2"] if g not in d.var_names]
assert not missing, f"panel genes missing: {missing}"
vals = d.X.data if sp.issparse(d.X) else np.asarray(d.X).ravel()
assert np.allclose(vals, np.round(vals)), "X should hold raw integer counts"
print(d.obs["BatchID"].value_counts().to_string())
print("structure OK")

GROUPS = {"ct1_mo": ("Cancer Cells (Mo)", "ct1"), "ct2_mo": ("Fibroblasts (Mo)", "ct2"),
          "ct1_co": ("Cancer Cells (Co)", "ct1"), "ct2_co": ("Fibroblasts (Co)", "ct2")}

In [ ]:
import matplotlib.pyplot as plt
from qxct.encoding import create_joint_histogram
from qxct.plotting import data_histogram


def fingerprint(adata, group_col, counts):
    """Per group: number of cells, total raw counts, and the joint histogram of the
    QuantumXCT gene panel (gene on = count > 0), i.e. exactly the states the circuits encode."""
    rows = []
    for key, (label, ct) in GROUPS.items():
        cells = (adata.obs[group_col].astype(str) == label).values
        genes = PANEL[ct]
        sub = counts[cells]
        M = sub[:, [adata.var_names.get_loc(g) for g in genes]]
        M = M.toarray() if sp.issparse(M) else np.asarray(M)
        rows.append({"group": key, "item": "n_cells", "value": int(cells.sum())})
        rows.append({"group": key, "item": "total_counts", "value": int(round(float(sub.sum())))})
        hist = dict(create_joint_histogram((M > 0).astype(int))[0])
        for i in range(2 ** len(genes)):
            s = format(i, f"0{len(genes)}b")
            rows.append({"group": key, "item": s, "value": int(hist.get(s, 0))})
    return pd.DataFrame(rows)


def plot_histograms(fp, name):
    """2x2 joint histograms: rows = mono / co-culture, columns = cell type 1 / 2."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for r, cond in enumerate(("mo", "co")):
        for c, ct in enumerate(("ct1", "ct2")):
            key = f"{ct}_{cond}"
            h = fp[(fp.group == key) & ~fp.item.isin(["n_cells", "total_counts"])]
            data_histogram(axes[r, c], dict(zip(h.item, h.value)), len(PANEL[ct]),
                           f"{name} - {GROUPS[key][0]}  ({' '.join(PANEL[ct])})")
    fig.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_data_hist_grid.png")
    fig.savefig(path, dpi=150)
    plt.show()
    print("saved", path)

In [ ]:
fp = fingerprint(d, "BatchID", d.X)
plot_histograms(fp, "ovarian")

In [ ]:
ref = pd.read_csv(REFERENCE)
cmp = fp.merge(ref, on=["group", "item"], how="outer")
same = int((cmp["value"] == cmp["reference"]).sum())
print(f"identical to the stored reference: {same}/{len(cmp)} values")
assert same == len(cmp), cmp[cmp["value"] != cmp["reference"]]
print("ovarian dataset verified")

## Optional: re-export the stored annotations

Only if you edit the annotated object: writes `data_prep/dataset_co_mo2_cell_metadata.csv`
and `data_prep/dataset_co_mo2_genes.txt` from `dataset_co_mo2.h5ad`.

In [ ]:
EXPORT_ANNOTATIONS = False
if EXPORT_ANNOTATIONS:
    A = ad.read_h5ad(OUT, backed="r")
    obs = A.obs.copy()
    obs.index.name = "barcode"
    obs.to_csv(META)
    with open(GENES, "w") as f:
        f.write("\n".join(map(str, A.var_names)))
    print(f"wrote {META} ({len(obs)} cells) and {GENES} ({A.n_vars} genes)")